# t5-battery: battery zero-shot 결과 확인

T5와 Chronos-2의 CALCE 재평가는 64→16·stride 1이며, 같은 배터리의 시간순 관측을 파일 경계를 넘어 사용한다. 누락·불완전 관측은 계속 제외한다. `SCOPE="calce64-16"`은 새 CALCE 실험, `pilot`은 일부 배터리 비교, `full`은 기존 C2 전체 결과(CALCE 32→8)다. 이 노트북은 저장된 결과만 읽으며 모델 추론을 자동 실행하지 않는다.

전체·초·중·후반 모두 Mean window R²를 사용한다. 각 창의 표준 R²를 단순평균한 뒤 배터리를 동일 가중 평균한다. 모델 간 비교는 `results/t5-c2-battery-comparison/t5-c2-battery_summary.csv`에 있다.

CALCE 실행: `python scripts/run_calce_zeroshot.py --model both`. Pilot 실행: `python scripts/run_battery_comparison.py --model both`. [조건과 파일 안내](../docs/t5-c2-battery.md).

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / "src/battery_model_zeroshot.py").is_file())
MODEL = "t5-battery"
DATASET = "calce_cs2_1c"  # pilot/full에서는 다른 데이터셋을 선택할 수 있음
SCOPE = "calce64-16"
RUN = PROJECT_ROOT / "results" / MODEL / SCOPE / DATASET
print("Result:", RUN)
print("Config:", PROJECT_ROOT / "configs" / (MODEL + ".json"))
manifest = RUN / (MODEL + "_run.json")
if manifest.is_file():
    settings = json.loads(manifest.read_text(encoding="utf-8"))
    print("Status:", settings["status"], "completed batteries:", settings.get("batteries_completed", 0))
    print("Device:", settings["device"])
else:
    print("아직 이 그룹의 실행이 시작되지 않았습니다.")

## 구간별 점수
관측 cycle 범위를 3등분하고 예측 시작 cycle로 구간을 정한다. R²는 한 예측창의 실제 미래 평균으로 계산한 뒤 해당 구간/전체의 창별 점수를 단순평균하고 배터리를 동일 가중 평균한다. 상수 창은 NaN으로 제외하고 음수는 유지한다. r2_n_windows·r2_undefined_windows로 유효·제외 창 수를 확인한다.

In [ ]:
for suffix in ["summary.csv", "battery_metrics.csv"]:
    path = RUN / (MODEL + "_" + suffix)
    if path.is_file():
        table = pd.read_csv(path, keep_default_na=False, na_values=[""])
        display(table.style.format({"picp80": "{:.1%}"}, na_rep="—"))

In [ ]:
coverage = RUN / (MODEL + "_coverage.csv")
if coverage.is_file():
    display(pd.read_csv(coverage, keep_default_na=False))

## 대표 SOH 곡선
각 구간 중앙에 가장 가까운 유효 창 3개를 표시한다. 그래프의 window R²와 구간 전체 점수는 집계 범위가 다르다. 겹치는 예측창은 독립 표본이 아니다. 그림은 그룹 평가 완료 후 생성된다.

In [ ]:
for figure in sorted(RUN.glob(MODEL + "_forecast_windows_*.png")):
    display(Image(filename=str(figure), width=1100))